In [1]:
%run "../Model Architecture/GPT2Model.ipynb"

# the-verdict is only 5,145 tokens: context 1024 leaves 0 val samples, 256 gives 18 train / 2 val
GPT_CONFIG_TRAIN = {**GPT_CONFIG_124M, "context_length": 256}

Ex.1 Inputs Shape:  torch.Size([1024])
Ex.1 Targets Shape: torch.Size([1024])
1st Batch Inputs shape:  torch.Size([4, 1024])
1st Batch Targets shape: torch.Size([4, 1024])
tensor([[3666, 1438,  318]])


In [2]:
import torch
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

start_context = "Hello, how are"
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
device

device(type='cpu')

In [3]:
def text_to_token_ids(text, tokenizer):
  encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor = torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor

text_to_token_ids(start_context, tokenizer)

tensor([[15496,    11,   703,   389]])

In [4]:
def token_ids_to_text(token_ids, tokenizer):
  flat = token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())

token_ids_to_text(text_to_token_ids(start_context, tokenizer), tokenizer)

'Hello, how are'

In [5]:
def calc_loss_batch(input_batch, target_batch, model, device):
  input_batch = input_batch.to(device)
  target_batch = target_batch.to(device)
  logits = model(input_batch)
  loss = torch.nn.functional.cross_entropy(
      logits.flatten(0,1), target_batch.flatten())

  return loss


In [6]:
def calc_loss_loader(data_loader, model, device, num_batches = None):

  total_loss = 0

  if len(data_loader) == 0:
    return float("nan")
  elif num_batches is None:
    num_batches = len(data_loader)
  else:
    num_batches = min(num_batches, len(data_loader))

  for i, (input_batch, target_batch) in enumerate(data_loader):
    if i < num_batches:
      loss = calc_loss_batch(
          input_batch, target_batch, model, device)
      total_loss += loss.item()
    else:
      break

  return total_loss/num_batches

In [7]:
def generate(model, idx, max_new_tokens, context_length,
                      temperature = 0.0, top_k = None,  eos_id = None):

  for i in range(max_new_tokens):
    idx_cond = idx[:, -context_length:]
    with torch.no_grad():
     logits = model(idx_cond)

    logits = logits[:,-1,:]

    if top_k is not None:
      topk_values, _ = torch.topk(logits, top_k)
      min_val = topk_values[:,-1:]
      logits = torch.where(
          condition = logits < min_val,
          input = torch.tensor(float("-inf")).to(logits.device),
          other = logits)

    if temperature > 0.0:
      logits = logits/temperature
      probs = torch.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples = 1)

    else:
      idx_next = torch.argmax(logits, dim = -1, keepdim = True)

    if idx_next == eos_id:
      break

    idx = torch.cat((idx, idx_next), dim = 1)

  return idx

In [8]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
  model.eval()

  with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches = eval_iter)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches = eval_iter)

  model.train()

  return train_loss, val_loss

In [9]:
def generate_and_print_sample(model, tokenizer, device, start_context ):

  model.eval()
  context_length = model.pos_emb.weight.shape[0]
  encoded = text_to_token_ids(start_context, tokenizer).to(device)
  with torch.no_grad():
    token_ids = generate(model = model, idx = encoded, 
                         max_new_tokens = 50, context_length = context_length)

  decoded_text = token_ids_to_text(token_ids, tokenizer)
  print(decoded_text.replace("\n"," "))
  model.train()



In [10]:
def train_model_simple(model, tokenizer, optimizer, device,
                       eval_freq, eval_iter,
                       num_epoch, start_context,
                       train_loader, val_loader):

  train_losses, val_losses, track_tokens_seen = [], [], []
  tokens_seen, global_step = 0, -1

  for epoch in range(num_epoch):
    model.train()

    for input_batch, target_batch in train_loader:

      optimizer.zero_grad()
      loss = calc_loss_batch(input_batch, target_batch, model, device)
      loss.backward()
      optimizer.step()

      global_step += 1
      tokens_seen +=  input_batch.numel()


      if global_step % eval_freq == 0:
        train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)

        print(f"Ep {epoch+1} (Step {global_step:06d}): "
                f"Train Loss {train_loss:.3f}, "
                f"Val Loss {val_loss:.3f}")


    generate_and_print_sample(model, tokenizer, device, start_context)

  return train_losses, val_losses, track_tokens_seen




In [11]:
torch.manual_seed(123)  # same random start every run, so results can be compared

# 1. split the story: 90% for training, 10% for validation (checking)
split_idx  = int(0.9 * len(text))
train_data = text[:split_idx]
val_data   = text[split_idx:]

# 2. data loaders (stride = context length, so the samples don't overlap)
ctx = GPT_CONFIG_TRAIN["context_length"]
train_loader = create_dataloader(train_data, batch_size=2, stride=ctx, config=GPT_CONFIG_TRAIN)
val_loader   = create_dataloader(val_data,   batch_size=2, stride=ctx, config=GPT_CONFIG_TRAIN,
                                 shuffle=False, drop_last=False)

# 3. model + optimizer
model     = GPT2Model(GPT_CONFIG_TRAIN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

# 4. train
train_losses, val_losses, tokens_seen = train_model_simple(
    model, tokenizer, optimizer, device,
    eval_freq=5, eval_iter=5, num_epoch=10,
    start_context=start_context,
    train_loader=train_loader, val_loader=val_loader)

Ep 1 (Step 000000): Train Loss 9.781, Val Loss 9.933
Ep 1 (Step 000005): Train Loss 8.111, Val Loss 8.339
Hello, how are,,,,,,,,,,,,.                                     
Ep 2 (Step 000010): Train Loss 6.661, Val Loss 7.048
Ep 2 (Step 000015): Train Loss 5.961, Val Loss 6.616
Hello, how are, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and,, and, and,
Ep 3 (Step 000020): Train Loss 5.726, Val Loss 6.600
Ep 3 (Step 000025): Train Loss 5.201, Val Loss 6.348
Hello, how are, and I had been. Gisburn, and I had.                                    
Ep 4 (Step 000030): Train Loss 4.417, Val Loss 6.278
Ep 4 (Step 000035): Train Loss 4.069, Val Loss 6.226
Hello, how are the               "I he had been--and it's the and he had been his pictures I had been the donkey of the donkey, I felt and down, and he had been
Ep 5 (Step 000040): Train Loss 3.732, Val Loss 6.160
Hello, how are the  "Oh, and he was to the fact by the last--his, and I 

# Loading and saving model and optimizer weights in PyTorch 

In [12]:
torch.save({
    "model":     model.state_dict(),
    "optimizer": optimizer.state_dict(),
}, "model_and_optimizer.pth")

In [14]:
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)

model = GPT2Model(GPT_CONFIG_TRAIN).to(device)  # same config as when saved, or the load fails
model.load_state_dict(checkpoint["model"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer"])

model.train();  # continue training; use model.eval() to only generate text